# E4, E6, E7, E8 — Experimentos locales (GPU)

## Preguntas
- **E4:** ¿una cabeza ordinal (CORN) reduce los errores lejanos vs softmax?
- **E6:** ¿un backbone de español social (RoBERTuito/BETO) supera al modelo genérico?
- **E7:** ¿conservar emojis/puntuación mejora vs la limpieza agresiva?
- **E8:** ¿generar positivos sintéticos con LLM sube las clases escasas 4/5?

## Método
Todo bajo el MISMO StratifiedKFold, semilla y protocolo de P0, variando UN factor por experimento (para no confundir variables). Bootstrap pareado contra el baseline.

> Para RE-EJECUTAR: `driver_local.py all` (E6,E7), `exp_e4.py` (E4), `exp_e8.py` (E8). Requieren GPU.

In [ ]:
# --- Configuracion comun ---
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import common as C
R = C.RESULTS
def load(f): return json.load(open(R / f))

# Patron de dos niveles: por defecto CARGA resultados ya calculados (segundos, sin GPU).
# Para RE-EJECUTAR desde cero (requiere GPU/Bedrock), pon RECOMPUTE=True.
RECOMPUTE = False

## E4 — Cabeza ordinal CORN (métrica primaria: MAE)

In [ ]:
e4 = load("e4_corn.json"); p0 = load("p0_stratified.json"); d = e4["delta_mae_neg_corn_minus_base"]
print(f"softmax P0: MAE {p0['oof_global']['mae']:.3f}   CORN: MAE {e4['corn_metrics']['mae']:.3f}")
print(f"Reduccion de MAE = {d['mean']:+.3f}  IC95 [{d['ci_low']:+.3f}, {d['ci_high']:+.3f}]  -> IC excluye 0 => SIGNIFICATIVO")

## E6 — Backbone

In [ ]:
e6 = load("e6_backbone.json")
pd.DataFrame([[k, round(v["oof_global"]["qwk"],3)] for k,v in e6.items() if "oof_global" in v],
             columns=["backbone","QWK"]).sort_values("QWK", ascending=False)

**RoBERTuito** (preentrenado en tweets en español) gana. **BETO queda por debajo** del baseline: estar en español no basta, importa el dominio social.

## E7 — Limpieza A/B

In [ ]:
e7 = load("e7_limpieza.json"); d = e7["delta_qwk_minima_minus_agresiva"]
print(f"agresiva QWK {e7['text_agresiva']['oof_global']['qwk']:.3f} | minima QWK {e7['text_minima']['oof_global']['qwk']:.3f}")
print(f"Delta {d['mean']:+.3f} IC[{d['ci_low']:+.3f},{d['ci_high']:+.3f}] -> dentro del ruido (no concluyente)")

## E8 — Augmentation clases 4/5

In [ ]:
e8 = load("e8_augmentation.json"); d = e8["delta_qwk_vs_p0"]
print(f"F1 clases 4/5: {e8['f1_clases45_base']:.3f} -> {e8['f1_clases45_aug']:.3f} (con +{e8['n_synthetic']} sinteticos)")
print(f"Delta QWK = {d['mean']:+.3f} IC[{d['ci_low']:+.3f},{d['ci_high']:+.3f}] -> SIGNIFICATIVO")

## Veredicto
Tres mejoras reales: **E4** (MAE, significativo), **E8** (clases raras, significativo), **E6** (RoBERTuito, al borde). **E7** dentro del ruido. Combinar E6+E4+E8 es el trabajo futuro más prometedor.

## Amenaza
Los sintéticos de E8 vienen de un LLM: se añaden SOLO al train (el test es 100% humano) para no contaminar la evaluación.